In [229]:
##Code in this file has been modified from Boris Murmann's github below, and the open-source textbook from Harald Pretl and colleagues:
# https://github.com/bmurmann/Book-on-gm-ID-design/blob/main/starter_files_open_source_tools/gf180mcuD/techsweep_plots_from_mat.ipynb
# Murmann and his colleague have written a book on gm/id-based design, "Systematic Design of Analog CMOS Circuits" (2017)

#Pretl's Open-Source Book: https://iic-jku.github.io/analog-circuit-design/analog_circuit_design.pdf

# Copyright 2024 Harald Pretl
# Licensed under the Apache License, Version 2.0 (the “License”); you may not use this
# file except in compliance with the License. You may obtain a copy of the License at
# http://www.apache.org/licenses/LICENSE-2.0


import numpy as np
import scipy.constants as sc
import matplotlib.pyplot as plt
from pygmid import Lookup as lk

nfet = lk('../nfet_03v3.mat')
pfet = lk('../pfet_03v3.mat')
VDS1 = 1.65

In [230]:
# define the given parameters as taken from the specification table or inital guesses
c_load = 35e-12 #Expected given parasitics + breadboard mounting of device
gm_id_m34 = 7
gm_id_m56 = 4
gm_id_m9 = 12
#length in microns
l_m12 = 1       #gain transistors in diff pair
l_m34 = 1     #cascode for diff pair
l_m569 = 0.5      #PMOS active load (gain-limiting) and CS amp... chosen to be the same to simplify balance condition
l_m78 = 1        #ISS + NMOS current source for CS

#Target Specifications
f_gb = 5e6 # -3dB bandwidth of the voltage buffer
c_comp = c_load/20
slew_r = 14 #v/us
v_min = 1.35
v_max = 2.8
print(c_comp)

# Current Source Limitations
i_ss = (slew_r*10**6)*c_comp
i_casb = 5e-6              #bias current for cascode bias voltage generation
i_ss_tot = i_casb + i_ss
print(i_ss_tot)

1.75e-12
2.95e-05


In [231]:
gm_m12 = 2*np.pi*c_comp*f_gb 
gm_id_m12 = gm_m12 / (i_ss/2)
print("gm_m12 =", round(gm_m12*10**3,3), "ms")
print("gm_id_m12 =", round(gm_id_m12))

vgs_m12 = nfet.look_upVGS(GM_ID=gm_id_m12, L=l_m12, VDS=VDS1, VSB=0.0)
print("Vgs_m12 =", round(1*vgs_m12,3))
#first get width of 7
id_w_m12 = nfet.look_up('ID_W', GM_ID=gm_id_m12, L=l_m78, VDS=vgs_m12, VSB=0)
w_m12 = (i_ss/2) / id_w_m12
w_m12_round = max(round(w_m12*2)/2, 0.5)
print('M12 W =', round(w_m12, 2), 'um, rounded W =', w_m12_round, 'um')



vds_m7_min = v_min - vgs_m12
gm_id_m7 = 2/vds_m7_min
print("gm_id_m7 =", round(gm_id_m7,2))
gm_m7 = gm_id_m7*i_ss_tot
print("gm_m7 =", round(gm_m7*1000,3), "mS")

gm_m12 = 0.055 ms
gm_id_m12 = 4
Vgs_m12 = 1.079
M12 W = 1.08 um, rounded W = 1.0 um
gm_id_m7 = 7.37
gm_m7 = 0.217 mS


In [235]:
gm_m9 = gm_m12 * 9 # safety margin, apparently helpful around 10
print("gm_m9 =", round(gm_m9*1000,3), "mS")



i_do = round((gm_m9/gm_id_m9),5)
print("Ido =", i_do*10**6, "uA")

vgs_m569 = pfet.look_upVGS(GM_ID=gm_id_m9, L=l_m569, VDS=VDS1, VSB=0.0)
id_w_m9 = pfet.lookup('ID_W', GM_ID=gm_id_m9, L=l_m569, VDS=vgs_m569, VSB=0)
w_m9 = i_do / id_w_m9
w_m9_round = max(round(w_m9*2)/2, 0.5)
print('M9 W =', round(w_m9, 2), 'um, rounded W =', w_m9_round, 'um')


## Based on current requirement, scale m8 from m7
#first get width of 7
vgs_m78 = nfet.look_upVGS(GM_ID=gm_id_m7, L=l_m78, VDS=VDS1, VSB=0.0)
id_w_m7 = nfet.look_up('ID_W', GM_ID=gm_id_m7, L=l_m78, VDS=vgs_m78, VSB=0)
w_m7 = i_do / id_w_m7
w_m7_round = max(round(w_m7*2)/2, 0.5)
print('M7 W =', round(w_m7, 2), 'um, rounded W =', w_m7_round, 'um')

#Then scale 8
w_m8 = w_m7 * (i_do/i_ss_tot)
w_m8_round = max(round(w_m8*2)/2, 0.5)
print('M8 W =', round(w_m8, 1), 'um, rounded W =', w_m8_round, 'um')

#Balance conditions require sizing of m56
w_m56 = w_m7*w_m9/(2*w_m8)
w_m56_round = max(round(w_m56*2)/2, 0.5)
print('M5/6 W =', round(w_m56, 2), 'um, rounded W =', w_m56_round, 'um')
id_w_m56 = i_do/w_m56

#check gmid on m56
gm_id_m56 = pfet.lookup('GM_ID', ID_W=id_w_m56, L=l_m569, VDS=VDS1, VSB=0)
print("gm_id_m56 =", round(1*gm_id_m56,2))


gm_m9 = 0.495 mS
Ido = 40.0 uA
M9 W = 51.98 um, rounded W = 52.0 um
M7 W = 9.18 um, rounded W = 9.0 um
M8 W = 12.4 um, rounded W = 12.5 um
M5/6 W = 19.17 um, rounded W = 19.0 um
gm_id_m56 = 7.92


In [237]:
#M34: May come down to fq stuff
gm_m34 = gm_id_m34 * (i_ss/2)
vgs_m34 = nfet.look_upVGS(GM_ID=gm_id_m34, L=l_m34, VDS=VDS1, VSB=0.0)
id_w_m34 = nfet.lookup('ID_W', GM_ID=gm_id_m34, L=l_m34, VDS=vgs_m34, VSB=0)
w_m34 = (i_ss/2) / id_w_m34
w_m34_round = max(round(w_m34*2)/2, 0.5)
print('M3/4 W =', round(w_m34, 2), 'um, rounded W =', w_m34_round, 'um')


#Intrinsic gain of mirrors and diff pair
gm_gds_m12 = nfet.lookup('GM_GDS', GM_ID=gm_id_m12, L=l_m12, VDS=VDS1, VSB=0)
gm_gds_m34 = nfet.lookup('GM_GDS', GM_ID=gm_id_m34, L=l_m34, VDS=VDS1, VSB=0)
gm_gds_m569 = pfet.lookup('GM_GDS', GM_ID=gm_id_m56, L=l_m569, VDS=VDS1, VSB=0)

#get diff pair gds values
ro_m12 = gm_gds_m12 / gm_m12
ro_m34 = gm_gds_m34 / gm_m34 

ro_cascode = (gm_m12*ro_m12+1)*ro_m34+ro_m12
print("ro_cascode =", round(ro_cascode /1000,2),"kOhm"  )

#get current mirror ro
gm_m56 = gm_id_m56 * (i_ss/2)
print("gm_m569 =", round(gm_m56/1e-3, 4), 'mS')
ro_m56 = gm_gds_m569 / gm_m56

print("ro_m56 =", round(ro_m56/1000,2), 'kOhm')
a0 = gm_m12 * (ro_cascode*ro_m56)/(ro_cascode + ro_m56) 
print('a0 =', round(20*np.log10(a0), 1), 'dB')

#zero position
f_zero = (gm_m34*ro_m56)*(gm_m9/c_comp)/(2*np.pi)
f_nondom = gm_m34*gm_m9*ro_m56/(2*np.pi*c_load)
print("f_zero =", round(f_zero/10**6, 3), 'MHz')        #this seems not correct
print("f_nondom =", round(f_nondom/10**6, 3), 'MHz')    #this also seems not correct


gm_cgs_m56 = pfet.lookup('GM_CGS', GM_ID=gm_id_m56, L=l_m569, VDS=VDS1, VSB=0)
gm_cdd_m56 = pfet.lookup('GM_CDD', GM_ID=gm_id_m56, L=l_m569, VDS=VDS1, VSB=0)
gm_cdd_m34 = nfet.lookup('GM_CDD', GM_ID=gm_id_m34, L=l_m34, VDS=VDS1, VSB=0)
c_load_parasitic = abs(gm_m56/gm_cgs_m56) + abs(gm_m56/gm_cdd_m56) + abs(gm_m34/gm_cdd_m34)

f_mirror = gm_m56/(2*np.pi*(2*c_load_parasitic))
print("f_mirror,est =", round(f_mirror/10**6, 3), 'MHz')


M3/4 W = 2.53 um, rounded W = 2.5 um
ro_cascode = 1215353.62 kOhm
gm_m569 = 0.0971 mS
ro_m56 = 2289.1 kOhm
a0 = 42.0 dB
f_zero = 8833.073 MHz
f_nondom = 441.654 MHz
f_mirror,est = 508.215 MHz


In [234]:
gm_gds_m78 = nfet.lookup('GM_GDS', GM_ID=gm_id_m78, L=l_m78, VDS=VDS1, VSB=0)
gm_m8 = gm_id_m78 * i_do

gds_m78 = gm_m8/gm_gds_m78
gds_m9 = gm_m9/gm_gds_m569

a1 = gm_m9/(gds_m9 + gds_m78)
print('a1 =', round(20*np.log10(a1), 1), 'dB')
av = a1*a0
print('av =', round(20*np.log10(av), 2), 'dB')


# ro_m8 = 1/gds_m78 

# print("Dominant Pole frequency", round(fdom,1), "Hz")
# print("Miller Cap = ", round(c_comp*10**12,3), "pF")


# fdom = f_gb/(av)
# c_comp = 1/((2*np.pi)*ro_m56*gm_m9*ro_m8*fdom)

a1 = 56.7 dB
av = 105.78 dB
